# Lab 1 — MCP Server with LangGraph `create_react_agent`

> **Mode: DEMO in class** | Time: ~40 min

## What you will learn
- The **three MCP primitives** — Tool, Resource, Prompt — and why an agent
  usually needs all three to behave correctly in production.
- How to derive **MCP bearer tokens from a user session** so nothing is
  hard-coded (user-owned credentials).
- Two ways to attach MCP tools to a LangGraph `create_react_agent`:
  1. **Discovery first, then wrap** — call `list_tools()` and build
     `StructuredTool` wrappers dynamically.
  2. **Direct attach** — decorate handler functions with `@tool` and hand
     the list straight to `create_react_agent`.
- How to observe every run with **Langfuse**.

## The 3 primitives cheat-sheet

| Primitive | Purpose                      | Analogy                              |
|-----------|------------------------------|--------------------------------------|
| **Tool**      | *Perform* an action (call API, run SQL, send email) | A button the agent can press         |
| **Resource**  | *Provide* read-only context (schema, rules, docs)   | A document the agent can read        |
| **Prompt**    | *Shape* the response (template with placeholders)   | A fill-in-the-blank form             |

> If you drop **Resource**, the agent has no schema -> invents table/column names -> wrong SQL.
> If you drop **Prompt**, the agent answers in any format it likes -> hard to QA.
> If you drop **Tool**, the agent cannot touch real data -> hallucinates.

---


## Step 1 — Environment

We start Jupyter from the dedicated conda env (`conda env create -f environment.yml`)
then select the **Extensible Agents (conda)** kernel.


In [1]:
import os, sys, json
_cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "lib")):
    PROJECT_ROOT = _cwd
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# Ensure DB is available
from data import DB_PATH
if not os.path.exists(DB_PATH):
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "db"))
    from setup_database import create_database
    create_database()

print(f"Project root : {PROJECT_ROOT}")
print(f"Model        : {os.getenv('OPENAI_MODEL','gpt-4.1-nano')}")
print(f"Langfuse     : {'enabled' if os.getenv('LANGFUSE_ENABLED','').lower()=='true' else 'disabled'}")


Project root : /mnt/oldssd/home/pc/Documents/cole_labs/extensible_agents
Model        : gpt-4.1-nano
Langfuse     : enabled


## Step 2 — Identity, Grants, Credential Factory

Production agents never hold hard-coded secrets. The flow is:

```
User login  ─►  SessionToken
               (signed JWT-like artefact)
                      │
                      ▼
             CredentialFactory
                      │
           ┌──────────┴──────────┐
           ▼                     ▼
   MCP Bearer Token        A2A credentials
   (scoped to user)        (per agent)
```

Grants are decided by admins — in real life this is a ticket/approval flow
similar to "Claude asking you to authorise access to your GitHub".


In [2]:
from identity import IdentityProvider, GrantRegistry, CredentialFactory, seed_lab_users
from a2a_framework import A2AAuthProvider
from agent_builder import (get_or_build_analytics_mcp,
                            build_langchain_tools_from_mcp,
                            reset_builder_state)
reset_builder_state()

idp = IdentityProvider()
grants = GrantRegistry()
seed_lab_users(idp, grants)              # fake the admin "grant approval" step

cred_factory = CredentialFactory(idp, grants)
cred_factory.register_a2a_provider(A2AAuthProvider())

# Build the analytics MCP server once; this also registers its auth provider
mcp_server = get_or_build_analytics_mcp(cred_factory)

print("Registered users and grants:")
for uid in ["admin_thiem","analyst_duc","analyst_mai","viewer_nam"]:
    g = grants.get(uid)
    print(f"  {uid:13s}  agents={sorted(g.agent_access)}  "
          f"mcp_scopes={ {k:sorted(v) for k,v in g.mcp_scopes.items()} }")


Registered users and grants:
  admin_thiem    agents=['analytics_agent', 'inventory_agent', 'writer_agent']  mcp_scopes={'datatech-analytics-mcp': ['customers:read', 'products:read', 'revenue:read', 'sql:execute'], 'datatech-inventory-mcp': ['orders:read', 'products:read', 'sql:execute']}
  analyst_duc    agents=['analytics_agent', 'writer_agent']  mcp_scopes={'datatech-analytics-mcp': ['products:read', 'revenue:read', 'sql:execute']}
  analyst_mai    agents=['analytics_agent']  mcp_scopes={'datatech-analytics-mcp': ['products:read', 'revenue:read']}
  viewer_nam     agents=[]  mcp_scopes={'datatech-analytics-mcp': ['products:read']}


## Step 3 — Login as **admin** (default for Sections 4–5)

Every following section uses `admin_thiem` so we can focus on the MCP mechanics
without tripping on grant errors. Section 6 will try *other* logins to show
the credential-derivation effect.


In [3]:
session = idp.login("admin_thiem", "admin456")
print(f"Logged in as: {session.display_name}")
print(f"Session id  : {session.session_id[:12]}...  expires in "
      f"{int(session.expires_at - session.created_at)}s")

mcp_token = cred_factory.derive_mcp_token(session, "datatech-analytics-mcp")
print(f"Derived MCP token (for analytics MCP): ...{mcp_token[-8:]}")


Logged in as: Nguyen Ba Thiem (Admin)
Session id  : 7V0uA7bzgF0p...  expires in 3600s
Derived MCP token (for analytics MCP): ...eVgPzLgM


## Step 4 — The 3 MCP primitives, in detail

### 4a. TOOLS
Tools are functions the agent can **call**. Each tool declares a JSON-Schema
for its parameters and an `allowed_roles`/`required_scope` so the server
itself enforces access control.

> **Why this matters** — the description and JSON-Schema are what the LLM
> reads when picking a tool. Vague descriptions = wrong tool chosen.


In [4]:
for t in mcp_server.list_tools(mcp_token):
    print(f"- {t['name']}")
    print(f"   {t['description'][:110]}...")
    print(f"   params: {list(t['parameters']['properties'].keys())}")


- query_revenue
   Query monthly revenue by region_id (HN=Hanoi, HC=HCMC, DN=Da Nang) over a date range (YYYY-MM). Returns VND am...
   params: ['region_id', 'start_month', 'end_month']
- list_products
   List products; optionally filter by category (Laptop, Phone, Tablet, Accessories, Monitor)....
   params: ['category']
- execute_sql
   Execute a read-only SQL SELECT statement against the DataTech database. Allowed tables: regions, products, cus...
   params: ['query']


### 4b. RESOURCES — this is the primitive students miss most often
Resources are **read-only context**. The agent does not "call" a resource; we
read it once and paste it into the system prompt before any reasoning begins.

In this lab the server exposes three:

- `datatech://schema`   — the DB schema (tables, columns, types, row counts)
- `datatech://rules`    — business rules (KPI thresholds, formatting)
- `datatech://company`  — high-level company info

Why a resource and not a static string in the system prompt? Two reasons:

1. **Access control** — the schema needs `sql:execute` scope. A viewer logging
   in would see the prompt without a schema section.
2. **Freshness** — updating the DB schema updates the agent's context on the
   next session, no re-deploy.


In [5]:
# The agent_builder already reads all visible resources when composing the system prompt.
# Here we just inspect them:
for r in mcp_server.list_resources(mcp_token):
    content = mcp_server.read_resource(r["uri"], mcp_token)
    print(f"--- {r['uri']} --- ({r['name']})")
    print(content if isinstance(content, str) else content)
    print()


--- datatech://schema --- (Database Schema)
TABLE customers (8 rows):
    id                   TEXT PRIMARY KEY
    name                 TEXT NOT NULL
    email                TEXT NOT NULL
    phone                TEXT
    region_id            TEXT NOT NULL
    vip_level            TEXT
    created_at           TEXT NOT NULL

TABLE internal_config (5 rows):
    key                  TEXT PRIMARY KEY
    value                TEXT NOT NULL

TABLE orders (26 rows):
    id                   TEXT PRIMARY KEY
    customer_id          TEXT NOT NULL
    product_id           TEXT NOT NULL
    region_id            TEXT NOT NULL
    quantity             INTEGER NOT NULL
    total_amount         INTEGER NOT NULL
    order_date           TEXT NOT NULL
    status               TEXT NOT NULL

TABLE products (10 rows):
    id                   TEXT PRIMARY KEY
    name                 TEXT NOT NULL
    category             TEXT NOT NULL
    price                INTEGER NOT NULL
    cost               

### 4c. PROMPTS — templates to lock the output format
Prompts are parameterised templates the server hands back rendered. They
standardise *how* the agent should frame an answer.


In [6]:
for p in mcp_server.list_prompts():
    print(f"- {p['name']}: {p['description']}")

# Render one to see the effect
rendered = mcp_server.render_prompt(
    "revenue_analysis",
    region="Hanoi", region_id="HN",
    current_month="2025-03", prior_month="2025-02")
print("\nrevenue_analysis template rendered:\n")
print(rendered)


- revenue_analysis: Period-over-period revenue analysis template
- sql_analysis: Structured SQL exploration template
- full_report: Comprehensive business report template

revenue_analysis template rendered:

Analyse revenue for Hanoi (region_id=HN). Compare 2025-03 against 2025-02. Return a markdown table with columns Month | Revenue (VND) | Change % | Status. Flag drops > 10% as NEEDS ATTENTION; growth > 10% as STRONG GROWTH. End with a two-sentence executive summary.


## Step 5 — Attach the MCP tools to a LangGraph agent

There are two common ways to wire MCP tools into `create_react_agent`. Both
are shown here so you can pick the one that matches your codebase.

### Method A — Discovery-first (recommended for dynamic, user-scoped agents)
Ask the MCP server *which* tools the current token may see, then wrap each
one as a LangChain `StructuredTool`. The tool list reflects the user's scopes
automatically.


In [7]:
from tracing import get_langchain_handler
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

langfuse_handler = get_langchain_handler()
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL","gpt-4.1-nano"),
                  api_key=os.getenv("OPENAI_API_KEY"))

# Method A — discovery first, wrap, attach
lc_tools_A = build_langchain_tools_from_mcp(mcp_server, mcp_token)
print("Method A tools:", [t.name for t in lc_tools_A])


  [Langfuse] LangChain CallbackHandler enabled.
Method A tools: ['query_revenue', 'list_products', 'execute_sql']


In [8]:
# Compose a system prompt from the MCP Resources (the classic pattern):
sys_parts = [
    "You are a senior data analyst for DataTech Vietnam.",
    "The DataTech database is your ONLY source of truth. "
    "You MUST call a tool for ANY question about revenue, products, "
    "customers, or orders. Never answer from memory.",
    "Follow the Business Rules exactly when formatting output.",
    "",
]
for r in mcp_server.list_resources(mcp_token):
    content = mcp_server.read_resource(r["uri"], mcp_token)
    if isinstance(content, str):
        sys_parts += [f"## {r['name']}  ({r['uri']})", content, ""]
system_prompt_A = "\n".join(sys_parts)

agent_A = create_react_agent(model=llm, tools=lc_tools_A,
                              prompt=system_prompt_A,
                              name="analytics_agent")
if langfuse_handler:
    agent_A = agent_A.with_config({"callbacks": [langfuse_handler],
                                    "metadata": {"user_id": session.user_id,
                                                  "method": "A"}})
print("Agent A ready.")


Agent A ready.


/tmp/ipykernel_373810/985681023.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_A = create_react_agent(model=llm, tools=lc_tools_A,


### Method B — Direct `@tool` attach (concise, static tool set)
Sometimes you already have Python handler functions and just want them
wrapped. Decorate with `@tool` and pass the list directly.


In [9]:
from langchain_core.tools import tool

@tool
def query_revenue(region_id: str, start_month: str, end_month: str) -> str:
    """Query monthly revenue by region_id (HN/HC/DN) over a date range (YYYY-MM)."""
    return json.dumps(mcp_server.call_tool(
        "query_revenue",
        {"region_id": region_id, "start_month": start_month, "end_month": end_month},
        token=mcp_token,
    ), ensure_ascii=False, indent=2)

@tool
def list_products(category: str = "") -> str:
    """List products; pass an empty string for all."""
    args = {"category": category} if category else {}
    return json.dumps(mcp_server.call_tool("list_products", args, token=mcp_token),
                      ensure_ascii=False, indent=2)

@tool
def execute_sql(query: str) -> str:
    """Execute a read-only SQL SELECT query against the DataTech database."""
    return json.dumps(mcp_server.call_tool("execute_sql", {"query": query},
                                            token=mcp_token),
                      ensure_ascii=False, indent=2)

lc_tools_B = [query_revenue, list_products, execute_sql]
agent_B = create_react_agent(model=llm, tools=lc_tools_B,
                              prompt=system_prompt_A, name="analytics_agent_B")
if langfuse_handler:
    agent_B = agent_B.with_config({"callbacks": [langfuse_handler],
                                    "metadata": {"user_id": session.user_id,
                                                  "method": "B"}})
print("Agent B ready — tools attached directly via @tool.")


Agent B ready — tools attached directly via @tool.


/tmp/ipykernel_373810/1453953998.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_B = create_react_agent(model=llm, tools=lc_tools_B,


### Run both agents on the same question
Identical output proves the two wiring styles are interchangeable.


In [10]:
def ask(agent, q, label):
    print(f"\n{'='*60}\n[{label}] {q}\n{'-'*60}")
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(result["messages"][-1].content)

ask(agent_A, "What was Hanoi revenue in January 2025?", "Method A")



[Method A] What was Hanoi revenue in January 2025?
------------------------------------------------------------


The revenue for Hanoi in January 2025 was 94,440,000 VND.


In [11]:
ask(agent_B, "What was Hanoi revenue in January 2025?", "Method B")



[Method B] What was Hanoi revenue in January 2025?
------------------------------------------------------------
The revenue for Hanoi in January 2025 was 94,440,000 VND.


### A query that forces all 3 primitives to be used
To produce a correct KPI report the agent must:

- **read the Schema resource**   (know table/column names)
- **follow the Rules resource**  (NEEDS ATTENTION / STRONG GROWTH flags)
- **render the Prompt template** (use it as the question framing)
- **call the Tool** (`query_revenue` / `execute_sql`) to pull the numbers


In [12]:
rendered = mcp_server.render_prompt(
    "revenue_analysis",
    region="Hanoi", region_id="HN",
    current_month="2025-03", prior_month="2025-02")
ask(agent_A, rendered, "Prompt + Resource + Tool (Method A)")



[Prompt + Resource + Tool (Method A)] Analyse revenue for Hanoi (region_id=HN). Compare 2025-03 against 2025-02. Return a markdown table with columns Month | Revenue (VND) | Change % | Status. Flag drops > 10% as NEEDS ATTENTION; growth > 10% as STRONG GROWTH. End with a two-sentence executive summary.
------------------------------------------------------------
| Month     | Revenue (VND) | Change % | Status             |
|------------|----------------|----------|-------------------|
| 2025-02    | 39,960,000     | -        | -                 |
| 2025-03    | 50,470,000     | 26.23%   | STRONG GROWTH   |

**Executive Summary:**  
Revenue for Hanoi increased by 26.23% from February to March 2025, indicating strong growth. No attention flag is needed as the change is an upward trend exceeding 10%.


## Step 6 — Different logins, different agents

Here is the punchline of the whole credential story. We log in as **three
different users** and build an agent for each. Because the MCP token is
derived from the session, each agent automatically sees only what the user's
grants permit.

Expected outcomes:

| User           | Grant set                                          | Behaviour                                         |
|----------------|----------------------------------------------------|---------------------------------------------------|
| `admin_thiem`  | all scopes on both MCP servers                     | can run SQL, get revenue, list products           |
| `analyst_mai`  | `revenue:read`, `products:read` only               | revenue works; SQL requests fall back / refused   |
| `viewer_nam`   | `products:read` only                               | only `list_products`; revenue / SQL denied        |


In [13]:
from agent_builder import build_langchain_tools_from_mcp

def build_agent_for_session(idp_session):
    try:
        tk = cred_factory.derive_mcp_token(idp_session, "datatech-analytics-mcp")
    except PermissionError as e:
        return None, f"Cannot build MCP token: {e}", []
    # Compose a system prompt from whichever resources this user can see
    sp_parts = [
        "You are a data analyst for DataTech Vietnam.",
        "The DataTech database is your ONLY source of truth. "
        "You MUST call a tool for any data question.",
        "If no suitable tool is available, tell the user plainly.",
        "",
    ]
    for r in mcp_server.list_resources(tk):
        content = mcp_server.read_resource(r["uri"], tk)
        if isinstance(content, str):
            sp_parts += [f"## {r['name']}", content, ""]
    tools = build_langchain_tools_from_mcp(mcp_server, tk)
    ag = create_react_agent(model=llm, tools=tools,
                             prompt="\n".join(sp_parts),
                             name=f"agent_for_{idp_session.user_id}")
    if langfuse_handler:
        ag = ag.with_config({"callbacks": [langfuse_handler],
                              "metadata": {"user_id": idp_session.user_id}})
    return ag, tk, [t.name for t in tools]


question = "What was Hanoi revenue in January 2025?  If you cannot find it, say so."

for uid, pwd in [("admin_thiem","admin456"),
                  ("analyst_mai","mai123"),
                  ("viewer_nam","nam789")]:
    print(f"\n========== login: {uid} ==========")
    sess = idp.login(uid, pwd)
    ag, tk, names = build_agent_for_session(sess)
    if ag is None:
        print("  Cannot build agent:", tk)
        continue
    print(f"  tools for this user: {names}")
    try:
        r = ag.invoke({"messages":[{"role":"user","content":question}]})
        print(f"  answer: {r['messages'][-1].content}")
    except Exception as e:
        print(f"  runtime error: {e}")



========== login: admin_thiem ==========
  tools for this user: ['query_revenue', 'list_products', 'execute_sql']


/tmp/ipykernel_373810/481431682.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  ag = create_react_agent(model=llm, tools=tools,


  answer: The revenue for Hanoi in January 2025 was 94,440,000 VND.

========== login: analyst_mai ==========
  tools for this user: ['query_revenue', 'list_products']


/tmp/ipykernel_373810/481431682.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  ag = create_react_agent(model=llm, tools=tools,


  answer: Hanoi's revenue in January 2025 was 94,440,000 VND.

========== login: viewer_nam ==========
  tools for this user: ['list_products']


/tmp/ipykernel_373810/481431682.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  ag = create_react_agent(model=llm, tools=tools,


  answer: I have retrieved the list of products. However, I still need to access the sales and revenue data for Hanoi in January 2025 to answer your question. I'll now look for that specific sales data.
I do not have direct access to the sales or revenue data. You may need to refer to the sales records or financial reports stored in your company's database for that period.


### What just happened?

- **`admin_thiem`** has every scope, so the agent exposes `query_revenue`,
  `list_products` and `execute_sql`. It answers with the exact figure.
- **`analyst_mai`** has `revenue:read` + `products:read` — `execute_sql` is
  hidden from her agent. She can still answer via `query_revenue`.
- **`viewer_nam`** only has `products:read` — her agent has a single tool.
  It cannot answer the revenue question, so it tells the user so (exactly the
  safe fallback we requested).

No credentials were hard-coded. Each agent's capability set is a strict
projection of `User -> Grants -> Session -> MCP token -> visible tools`.


## Step 7 — Observability

Whether Langfuse is enabled or not, the MCP server keeps a structured call
log with correlation IDs, per-call duration, and the calling client.


In [14]:
print("MCP call log (last 6 entries):")
for e in mcp_server.get_call_log()[-6:]:
    print(f"  cid={e['correlation_id']}  {e['tool']:16s}  "
          f"client={e['client_id']:30s}  {e['duration_ms']:.1f} ms")

from tracing import flush
flush()


MCP call log (last 6 entries):
  cid=78530d06  query_revenue     client=admin_thiem@datatech-analytics-mcp  0.2 ms
  cid=0adb07d4  query_revenue     client=admin_thiem@datatech-analytics-mcp  1.0 ms
  cid=0a698208  query_revenue     client=admin_thiem@datatech-analytics-mcp  1.2 ms
  cid=557fc17a  query_revenue     client=admin_thiem@datatech-analytics-mcp  0.3 ms
  cid=55bdd99d  query_revenue     client=analyst_mai@datatech-analytics-mcp  0.2 ms
  cid=71de9cd3  list_products     client=viewer_nam@datatech-analytics-mcp  0.3 ms


## Takeaways

1. The **3 MCP primitives** — Tool, Resource, Prompt — serve different jobs.
   Omit Resource and the agent hallucinates; omit Prompt and the format drifts.
2. **User-owned credentials**: `login()` -> `SessionToken` -> `CredentialFactory`
   -> scoped MCP token. Agents are rebuilt per-session, not pre-configured.
3. `create_react_agent` accepts MCP tools whether you discover-and-wrap
   (Method A) or directly attach with `@tool` (Method B).
4. **Langfuse** traces attach via `create_react_agent(...).with_config(...)`.

**Lab 2** — add a **Skill** to the same agent and watch the output quality
jump without changing the tools.
